# 09. Итоговые таблицы для курсовой работы

Ноутбук формирует компактные сводные таблицы по дневной выборке: число наблюдений, число выпусков, период наблюдений, средние значения цены, доходности и объёма торгов.


## 1. Импорт и загрузка данных

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SUMMARY_DIR = DATA_DIR / "summary"
FIGURES_DIR = PROJECT_ROOT / "report" / "figures"

for path in [SUMMARY_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

In [2]:
DAILY_PATH = PROCESSED_DIR / "ofz_clean_daily_2016_2026.csv"
DATALENS_DAILY_PATH = PROCESSED_DIR / "ofz_for_datalens_daily_2016_2026.csv"

for path in [DAILY_PATH, DATALENS_DAILY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}. Сначала запустите предыдущие ноутбуки.")

df_daily = pd.read_csv(DAILY_PATH)
df_datalens = pd.read_csv(DATALENS_DAILY_PATH)

for df in [df_daily, df_datalens]:
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"], errors="coerce")

print("Daily clean shape:", df_daily.shape)
print("Daily DataLens shape:", df_datalens.shape)
display(df_daily.head())

Daily clean shape: (15522, 45)
Daily DataLens shape: (15522, 13)


,TRADEDATE,SECID,ISSUE_NAME,SHORTNAME,BOARDID,COUPON_TYPE,ANALYSIS_GROUP,MATURITY_GROUP,PLOT_GROUP,OPEN,HIGH,LOW,CLOSE,WAPRICE,PRICE_FOR_ANALYSIS,CLEAN_PRICE_RUB,DIRTY_PRICE_RUB,PRICE_DEV_FROM_100,PRICE_RETURN,YIELDATWAP,YIELDCLOSE,YIELD_FOR_ANALYSIS,VOLUME,VALUE,NUMTRADES,ACCINT,FACEVALUE,COUPONPERCENT,COUPONVALUE,MATDATE,YEARS_TO_MATURITY,DURATION,CLOSE_IQR_OUTLIER,PRICE_RETURN_IQR_OUTLIER,YIELD_IQR_OUTLIER,YIELD_EXTREME_LEVEL_FLAG,HARD_DATA_ERROR,ANY_OUTLIER_FLAG,OUTLIER_REPLACED_WITH_DAILY_MEDIAN,CLOSE_RAW,WAPRICE_RAW,PRICE_FOR_ANALYSIS_RAW,YIELDATWAP_RAW,YIELDCLOSE_RAW,YIELD_FOR_ANALYSIS_RAW
0,2016-05-04,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,96.0501,96.0501,95.2501,95.5993,95.5773,95.5773,955.993,973.183,-4.4007,NaN,8.99,8.98,8.99,790250,7.549269e+08,122,17.19,1000,NaN,NaN,2027-02-03,10.751540,2610.0,False,False,False,False,False,False,False,95.5993,95.5773,95.5773,8.99,8.98,8.99
1,2016-05-05,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.6000,95.8000,95.3550,95.6201,95.6335,95.6335,956.201,973.621,-4.3799,0.000218,8.98,8.98,8.98,118606,1.133353e+08,58,17.42,1000,NaN,NaN,2027-02-03,10.748802,2610.0,False,False,False,False,False,False,False,95.6201,95.6335,95.6335,8.98,8.98,8.98
2,2016-05-06,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8888,95.9000,95.7000,95.9000,95.7770,95.7770,959.000,976.640,-4.1000,0.002927,8.95,8.94,8.95,218341,2.091805e+08,28,17.64,1000,NaN,NaN,2027-02-03,10.746064,2610.0,False,False,False,False,False,False,False,95.9000,95.7770,95.7770,8.95,8.94,8.95
3,2016-05-10,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.8000,95.9500,95.2900,95.7000,95.7496,95.7496,957.000,975.530,-4.3000,-0.002086,8.96,8.97,8.96,27776,2.660816e+07,56,18.53,1000,NaN,NaN,2027-02-03,10.735113,2606.0,False,False,False,False,False,False,False,95.7000,95.7496,95.7496,8.96,8.97,8.96
4,2016-05-11,SU26207RMFS9,ОФЗ-ПД 26207,ОФЗ 26207,TQOB,fixed,short_fixed_selected,long,long,95.9499,96.5999,95.7600,96.4500,96.3739,96.3739,964.500,983.260,-3.5500,0.007837,8.86,8.85,8.86,555283,5.351472e+08,172,18.76,1000,NaN,NaN,2027-02-03,10.732375,2610.0,False,False,False,False,False,False,False,96.4500,96.3739,96.3739,8.86,8.85,8.86


## 2. Сводная таблица по всей выборке

In [3]:
sample_summary = pd.DataFrame([
    {
        "dataset": "daily",
        "observations": len(df_daily),
        "unique_bonds": df_daily["SECID"].nunique(),
        "date_min": df_daily["TRADEDATE"].min(),
        "date_max": df_daily["TRADEDATE"].max(),
        "avg_price": df_daily["PRICE_FOR_ANALYSIS"].mean(),
        "avg_yield": df_daily["YIELD_FOR_ANALYSIS"].mean(),
        "total_value": df_daily["VALUE"].sum(),
    }
])

display(sample_summary)

,dataset,observations,unique_bonds,date_min,date_max,avg_price,avg_yield,total_value
0,daily,15522,12,2016-05-04,2026-04-30,89.666594,9.815881,5.160706e+12


## 3. Сводная таблица по типам облигаций

In [4]:
type_summary = (
    df_daily
    .groupby("COUPON_TYPE", as_index=False)
    .agg(
        observations=("TRADEDATE", "count"),
        unique_bonds=("SECID", "nunique"),
        date_min=("TRADEDATE", "min"),
        date_max=("TRADEDATE", "max"),
        avg_price=("PRICE_FOR_ANALYSIS", "mean"),
        avg_yield=("YIELD_FOR_ANALYSIS", "mean"),
        total_value=("VALUE", "sum"),
    )
    .sort_values("COUPON_TYPE")
)

display(type_summary)

,COUPON_TYPE,observations,unique_bonds,date_min,date_max,avg_price,avg_yield,total_value
0,fixed,13751,9,2016-05-04,2026-04-30,88.852513,11.080075,5.055390e+12
1,floating,1771,3,2023-05-04,2026-04-30,95.987557,0.000000,1.053161e+11


## 4. Сводная таблица по выпускам

In [5]:
bond_summary = (
    df_daily
    .groupby(["SECID", "ISSUE_NAME", "COUPON_TYPE", "ANALYSIS_GROUP"], dropna=False, as_index=False)
    .agg(
        observations=("TRADEDATE", "count"),
        date_min=("TRADEDATE", "min"),
        date_max=("TRADEDATE", "max"),
        avg_price=("PRICE_FOR_ANALYSIS", "mean"),
        avg_yield=("YIELD_FOR_ANALYSIS", "mean"),
        total_value=("VALUE", "sum"),
    )
    .sort_values(["COUPON_TYPE", "SECID"])
)

display(bond_summary)

,SECID,ISSUE_NAME,COUPON_TYPE,ANALYSIS_GROUP,observations,date_min,date_max,avg_price,avg_yield,total_value
0,SU26207RMFS9,ОФЗ-ПД 26207,fixed,short_fixed_selected,2516,2016-05-04,2026-04-30,99.596049,9.715262,1.061034e+12
1,SU26219RMFS4,ОФЗ-ПД 26219,fixed,short_fixed_selected,2472,2016-07-06,2026-04-30,98.415282,9.733794,6.662570e+11
2,SU26226RMFS9,ОФЗ-ПД 26226,fixed,short_fixed_selected,1817,2019-02-06,2026-04-30,98.513641,10.381365,5.687586e+11
3,SU26232RMFS7,ОФЗ-ПД 26232,fixed,medium_fixed_selected,1602,2019-12-11,2026-04-30,88.568110,10.829607,3.774011e+11
4,SU26236RMFS8,ОФЗ-ПД 26236,fixed,medium_fixed_selected,1364,2020-11-25,2026-04-30,83.031208,11.722903,3.623349e+11
5,SU26238RMFS4,ОФЗ-ПД 26238,fixed,long_fixed_selected,1219,2021-06-23,2026-04-30,68.191039,12.118187,8.862868e+11
6,SU26240RMFS0,ОФЗ-ПД 26240,fixed,long_fixed_selected,1203,2021-07-15,2026-04-30,70.181092,12.366201,3.870257e+11
7,SU26242RMFS6,ОФЗ-ПД 26242,fixed,medium_fixed_selected,829,2023-01-26,2026-04-30,85.459359,13.880072,2.593300e+11
8,SU26243RMFS4,ОФЗ-ПД 26243,fixed,long_fixed_selected,729,2023-06-22,2026-04-30,76.002974,14.402428,4.869617e+11
9,SU29024RMFS5,ОФЗ-ПК 29024,floating,floating_coupon,762,2023-05-04,2026-04-30,96.065100,0.000000,6.460038e+10


## 5. Сохранение таблиц

In [6]:
sample_summary_path = SUMMARY_DIR / "sample_summary_table.csv"
type_summary_path = SUMMARY_DIR / "bond_type_summary_table.csv"
bond_summary_path = SUMMARY_DIR / "bond_issue_summary_table.csv"
latex_table_path = SUMMARY_DIR / "sample_summary_table.tex"

sample_summary.to_csv(sample_summary_path, index=False, encoding="utf-8-sig")
type_summary.to_csv(type_summary_path, index=False, encoding="utf-8-sig")
bond_summary.to_csv(bond_summary_path, index=False, encoding="utf-8-sig")
sample_summary.to_latex(latex_table_path, index=False, escape=False)

print("Saved files:")
print("-", sample_summary_path.resolve())
print("-", type_summary_path.resolve())
print("-", bond_summary_path.resolve())
print("-", latex_table_path.resolve())

Saved files:
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/sample_summary_table.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/bond_type_summary_table.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/bond_issue_summary_table.csv
- /Users/daniilsestov/Desktop/курсач для гита/ofz_git_notebooks_daily_only/data/summary/sample_summary_table.tex
